# Подбор модели 

В этом разделе строим несколько моделей арендной платы и выбираем итоговую спецификацию. Сначала сравниваю обычную линейную модель и лог-линейные модели, затем проверяю качество моделей, выбираю лучшую.

## 1. Подготовка данных для моделей

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.iolib.summary2 import summary_col
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, linear_reset
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import jarque_bera

Загружаю датасет

In [2]:
df = pd.read_excel("data.xlsx")

Задаю группы переменных для моделей.

In [3]:
base_vars = [
    'ln_total_meters',
    'rooms_count',
    'center_distance',
    'metro_time_min'
]

building_vars = [
    'relative_floor',
    'is_apartment'
]

rent_condition_vars = [
    'no_commission',
    'has_deposit',
    'utilities_included',
    'kids_allowed',
    'pets_allowed'
]

amenity_vars = [
    'has_furniture',
    'has_washing_machine',
    'has_dryer',
    'has_fridge',
    'has_tv',
    'has_conditioner',
    'has_dishwasher',
    'has_microwave',
    'has_boiler'
]

repair_vars = [
    'repair_designer',
    'repair_euro'
]

In [4]:
base_vars = [x for x in base_vars if x in df.columns]
building_vars = [x for x in building_vars if x in df.columns]
rent_condition_vars = [x for x in rent_condition_vars if x in df.columns]
amenity_vars = [x for x in amenity_vars if x in df.columns]
repair_vars = [x for x in repair_vars if x in df.columns]

print('base:', base_vars)
print('building:', building_vars)
print('rent:', rent_condition_vars)
print('amenities:', amenity_vars)
print('repair:', repair_vars)

base: ['ln_total_meters', 'rooms_count', 'center_distance', 'metro_time_min']
building: ['relative_floor', 'is_apartment']
rent: ['no_commission', 'has_deposit', 'utilities_included', 'kids_allowed', 'pets_allowed']
amenities: ['has_furniture', 'has_washing_machine', 'has_dryer', 'has_fridge', 'has_tv', 'has_conditioner', 'has_dishwasher', 'has_microwave', 'has_boiler']
repair: ['repair_designer', 'repair_euro']


Собираю отдельный датасет для регрессий и удаляю пропуски только по тем переменным, которые используются в моделях.

In [5]:
model_cols = ['price', 'ln_price'] + base_vars + building_vars + rent_condition_vars + amenity_vars + repair_vars

df_model = df[model_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
df_model['amenities_count'] = df_model[amenity_vars].sum(axis=1)

df_model.shape

(987, 25)

## 2. Спецификации моделей

Дальше оцениваем пять моделей.

M1 — базовая линейная модель для цены в рублях.

M2 — базовая лог-линейная модель.

M3 — расширенная лог-линейная модель с характеристиками здания и условиями аренды.

M4 — полная лог-линейная модель с отдельными признаками техники и ремонта.

M5  — альтернативная лог-линейная спецификация. В ней отдельные признаки техники заменены на агрегированный показатель amenities_count. Это позволяет сократить число переменных и снизить риск мультиколлинеарности между признаками оснащения квартиры.

In [6]:
formula_m1 = 'price ~ ' + ' + '.join(base_vars)

formula_m2 = 'ln_price ~ ' + ' + '.join(base_vars)

formula_m3 = 'ln_price ~ ' + ' + '.join(base_vars + building_vars + rent_condition_vars)

formula_m4 = 'ln_price ~ ' + ' + '.join(base_vars + building_vars + rent_condition_vars + amenity_vars + repair_vars)

formula_m5 = 'ln_price ~ ' + ' + '.join(base_vars + building_vars + rent_condition_vars + ['amenities_count'] + repair_vars)

print(formula_m1)
print(formula_m2)
print(formula_m3)
print(formula_m4)
print(formula_m5)

price ~ ln_total_meters + rooms_count + center_distance + metro_time_min
ln_price ~ ln_total_meters + rooms_count + center_distance + metro_time_min
ln_price ~ ln_total_meters + rooms_count + center_distance + metro_time_min + relative_floor + is_apartment + no_commission + has_deposit + utilities_included + kids_allowed + pets_allowed
ln_price ~ ln_total_meters + rooms_count + center_distance + metro_time_min + relative_floor + is_apartment + no_commission + has_deposit + utilities_included + kids_allowed + pets_allowed + has_furniture + has_washing_machine + has_dryer + has_fridge + has_tv + has_conditioner + has_dishwasher + has_microwave + has_boiler + repair_designer + repair_euro
ln_price ~ ln_total_meters + rooms_count + center_distance + metro_time_min + relative_floor + is_apartment + no_commission + has_deposit + utilities_included + kids_allowed + pets_allowed + amenities_count + repair_designer + repair_euro


## 3. Оценивание моделей

In [7]:
model_1 = smf.ols(formula_m1, data=df_model).fit()
model_2 = smf.ols(formula_m2, data=df_model).fit()
model_3 = smf.ols(formula_m3, data=df_model).fit()
model_4 = smf.ols(formula_m4, data=df_model).fit()
model_5 = smf.ols(formula_m5, data=df_model).fit()

In [8]:
models_table = summary_col(
    [model_1, model_2, model_3, model_4, model_5],
    model_names=['M1 price', 'M2 log', 'M3 extended', 'M4 full', 'M5 amenities'],
    stars=True,
    float_format='%.3f',
    info_dict={
        'N': lambda x: f'{int(x.nobs)}',
        'R2': lambda x: f'{x.rsquared:.3f}',
        'Adj. R2': lambda x: f'{x.rsquared_adj:.3f}',
        'AIC': lambda x: f'{x.aic:.1f}',
        'BIC': lambda x: f'{x.bic:.1f}',
        'F p-value': lambda x: f'{x.f_pvalue:.3e}'
    }
)

models_table

,M1 price,M2 log,M3 extended,M4 full,M5 amenities
Intercept,-231793.856***,10.029***,10.012***,10.499***,10.221***
,(43618.366),(0.170),(0.199),(0.231),(0.216)
ln_total_meters,96440.132***,0.316***,0.299***,0.283***,0.289***
,(13618.683),(0.053),(0.054),(0.054),(0.054)
rooms_count,26123.120***,0.256***,0.232***,0.207***,0.228***
,(8113.996),(0.032),(0.032),(0.033),(0.032)
center_distance,-3252.130***,-0.014***,-0.012***,-0.009***,-0.011***
,(712.806),(0.003),(0.003),(0.003),(0.003)
metro_time_min,-757.314,-0.003,-0.002,-0.001,-0.002
,(627.227),(0.002),(0.002),(0.002),(0.002)


In [11]:
models = {
    'M1 price': model_1,
    'M2 log': model_2,
    'M3 extended': model_3,
    'M4 full': model_4,
    'M5 amenities': model_5,
}

model_comparison = pd.DataFrame({
    name: {
        'N': int(model.nobs),
        'R2': model.rsquared,
        'Adj. R2': model.rsquared_adj,
        'AIC': model.aic,
        'BIC': model.bic,
        'F p-value': model.f_pvalue
    }
    for name, model in models.items()
}).T

model_comparison.round(4).T

,M1 price,M2 log,M3 extended,M4 full,M5 amenities
N,987.0000,987.0000,987.0000,987.0000,987.0000
R2,0.3643,0.5043,0.5226,0.5383,0.5265
Adj. R2,0.3617,0.5022,0.5172,0.5278,0.5197
AIC,26156.0968,1567.3675,1544.2276,1533.0822,1542.0711
BIC,26180.5702,1591.8408,1602.9637,1645.6596,1615.4911
F p-value,0.0000,0.0000,0.0000,0.0000,0.0000


## Модель 6

Сделаю финальную лучшую модель номер 6, которая будет включать важнейшие значимые переменные, удобна для интерпретации и не перегружена.

- В предыдущих спецификациях было видно, что полная модель M4 даёт высокое качество, но содержит слишком много отдельных признаков техники, часть из которых незначима и может мультиколлинеарность.
- Модель M5 показала, что вместо отдельных признаков техники можно использовать показатель `amenities_count`. Но в ней всё ещё оставались незначимые переменные,особенно напрягает незначимость `amenities_count`.
- Поэтому в M6 я оставляю только основные факторы, которые либо стабильно значимы, либо важны содержательно: площадь, число комнат, расстояние до центра, относительный этаж, тип объекта, коммунальные платежи, ремонт и количество удобств.


In [9]:
formula_m6 = '''
ln_price ~ ln_total_meters
         + rooms_count
         + center_distance
         + relative_floor
         + is_apartment
         + utilities_included
         + repair_euro
         + amenities_count
'''

model_6 = smf.ols(formula_m6, data=df_model).fit()
print(model_6.summary())

                            OLS Regression Results                            
Dep. Variable:               ln_price   R-squared:                       0.523
Model:                            OLS   Adj. R-squared:                  0.519
Method:                 Least Squares   F-statistic:                     134.1
Date:                Thu, 07 May 2026   Prob (F-statistic):          1.56e-151
Time:                        19:20:14   Log-Likelihood:                -759.53
No. Observations:                 987   AIC:                             1537.
Df Residuals:                     978   BIC:                             1581.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             10.3224      0

In [ ]:
compare_final = pd.DataFrame({
    'M4_full': {
        'R2': model_4.rsquared,
        'Adj_R2': model_4.rsquared_adj,
        'AIC': model_4.aic,
        'BIC': model_4.bic
    },
    'M5_amenities': {
        'R2': model_5.rsquared,
        'Adj_R2': model_5.rsquared_adj,
        'AIC': model_5.aic,
        'BIC': model_5.bic
    },
    'M6_final': {
        'R2': model_6.rsquared,
        'Adj_R2': model_6.rsquared_adj,
        'AIC': model_6.aic,
        'BIC': model_6.bic
    }
}).T

compare_final.round(4)

,R2,Adj_R2,AIC,BIC
M4_full,0.5383,0.5278,1533.0822,1645.6596
M5_amenities,0.5265,0.5197,1542.0711,1615.4911
M6_final,0.5231,0.5192,1537.0579,1581.1099


Итог: дальше я буду использовать M6 как основную модель, потому что она достаточно хорошо объясняет цену аренды, не перегружена лишними переменными и лучше подходит для проверки предпосылок и содержательной интерпретации.


##  Проверка предпосылок

### 1. Линейность моделей по параметрам

В нашей работе все построенные модели удовлетворяют этой предпосылке, все модели линейные, либо лог-линейные.

Хотя в моделях используются логарифмы переменных и бинарные признаки, сами коэффициенты входят в уравнения линейно. Поэтому предпосылка линейности по параметрам выполняется для всех моделей. Также выписываю все формулы:

In [ ]:
print('M1:', formula_m1)
print('M2:', formula_m2)
print('M3:', formula_m3)
print('M4:', formula_m4)
print('M5:', formula_m5)
print('M6:', formula_m6)

M1: price ~ ln_total_meters + rooms_count + center_distance + metro_time_min
M2: ln_price ~ ln_total_meters + rooms_count + center_distance + metro_time_min
M3: ln_price ~ ln_total_meters + rooms_count + center_distance + metro_time_min + relative_floor + is_apartment + no_commission + has_deposit + utilities_included + kids_allowed + pets_allowed
M4: ln_price ~ ln_total_meters + rooms_count + center_distance + metro_time_min + relative_floor + is_apartment + no_commission + has_deposit + utilities_included + kids_allowed + pets_allowed + has_furniture + has_washing_machine + has_dryer + has_fridge + has_tv + has_conditioner + has_dishwasher + has_microwave + has_boiler + repair_designer + repair_euro
M5: ln_price ~ ln_total_meters + rooms_count + center_distance + metro_time_min + relative_floor + is_apartment + no_commission + has_deposit + utilities_included + kids_allowed + pets_allowed + amenities_count + repair_designer + repair_euro
M6: 
ln_price ~ ln_total_meters
         + roo

### 2. Случайность и независимость наблюдений

Следующая предпосылка теоремы Гаусса–Маркова - наблюдения должны быть случайными и независимыми друг от друга.

Наблюдения можно считать независимыми, так как данные являются пространственной выборкой: одно наблюдение соответствует одному объявлению о квартире.

### 3. Проверка мультиколлинеарности

Для проверки мультиколлинеарности используем показатель VIF, если он больше 10, то это может указывать на сильную мультиколлинеарность.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

models = {
    'M1 price': model_1,
    'M2 log': model_2,
    'M3 log+': model_3,
    'M4 full': model_4,
    'M5 amenities': model_5,
    'M6 final': model_6
}

vif_tables = {}

for name, model in models.items():
    X = pd.DataFrame(model.model.exog, columns=model.model.exog_names)

    vif_table = pd.DataFrame({
        'variable': X.columns,
        'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    })

    vif_tables[name] = vif_table.sort_values('VIF', ascending=False).round(3)

In [ ]:
vif_summary = []

for model_name, model in models.items():
    X = pd.DataFrame(model.model.exog, columns=model.model.exog_names)

    vif_table = pd.DataFrame({
        'variable': X.columns,
        'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    })

    vif_table = vif_table[vif_table['variable'] != 'Intercept']

    max_row = vif_table.loc[vif_table['VIF'].idxmax()]

    vif_summary.append({
        'Model': model_name,
        'Max VIF': max_row['VIF'],
        'Var with max VIF': max_row['variable'],
        'Multicol problem': 'Yes' if max_row['VIF'] > 10 else 'No'
    })

vif_summary = pd.DataFrame(vif_summary)

vif_summary['Max VIF'] = vif_summary['Max VIF'].round(3)

vif_summary

,Model,Max VIF,Var with max VIF,Multicol problem
0,M1 price,4.555,ln_total_meters,No
1,M2 log,4.555,ln_total_meters,No
2,M3 log+,4.804,ln_total_meters,No
3,M4 full,36.765,has_washing_machine,Yes
4,M5 amenities,4.826,ln_total_meters,No
5,M6 final,4.759,ln_total_meters,No


In [ ]:
for name, table in vif_tables.items():
    print(name)
    display(table)

M1 price


,variable,VIF
0,Intercept,99.829
1,ln_total_meters,4.555
2,rooms_count,4.534
3,center_distance,1.147
4,metro_time_min,1.068


M2 log


,variable,VIF
0,Intercept,99.829
1,ln_total_meters,4.555
2,rooms_count,4.534
3,center_distance,1.147
4,metro_time_min,1.068


M3 log+


...

,variable,VIF
0,Intercept,195.203
13,has_washing_machine,36.765
15,has_fridge,34.639
2,rooms_count,5.110
1,ln_total_meters,4.912
8,has_deposit,4.221
16,has_tv,3.954
22,repair_euro,3.747
21,repair_designer,3.699
19,has_microwave,1.821


M5 amenities


,variable,VIF
0,Intercept,166.807
1,ln_total_meters,4.826
2,rooms_count,4.814
14,repair_euro,3.590
13,repair_designer,3.578
12,amenities_count,3.021
8,has_deposit,3.000
10,kids_allowed,1.435
11,pets_allowed,1.402
3,center_distance,1.204


M6 final


,variable,VIF
0,Intercept,127.592
1,ln_total_meters,4.759
2,rooms_count,4.645
8,amenities_count,1.492
6,utilities_included,1.182
3,center_distance,1.162
7,repair_euro,1.105
5,is_apartment,1.096
4,relative_floor,1.020


У M4 есть VIF больше 36, значит это мультиколлинеарность.
В остальных, в том числе М6, нет мультиколлинеарности.

### 5. Гетероскедастичность

Посмотрим 2 теста Бреуша–Пагана и Уайта, смотрим на  значение — `F p-value`.

Гипотезы:

H_0: дисперсия случайной ошибки постоянна, то есть есть гомоскедастичность: p-value ≥ 0.05.

H_1: дисперсия случайной ошибки непостоянна, то есть есть гетероскедастичность: p-value < 0.05.

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan, het_white

models = {
    'M1 price': model_1,
    'M2 log': model_2,
    'M3 log+': model_3,
    'M4 full': model_4,
    'M5 amenities': model_5,
    'M6 final': model_6
}

hetero_results = []

for name, model in models.items():
    bp = het_breuschpagan(model.resid, model.model.exog)
    white = het_white(model.resid, model.model.exog)

    hetero_results.append({
        'model': name,
        'BP LM p-value': bp[1],
        'BP F p-value': bp[3],
        'White LM p-value': white[1],
        'White F p-value': white[3],
        'BP hetero': bp[3] < 0.05,
        'White hetero': white[3] < 0.05
    })

hetero_table = pd.DataFrame(hetero_results)

hetero_table.round(5)

,model,BP LM p-value,BP F p-value,White LM p-value,White F p-value,BP hetero,White hetero
0,M1 price,0.00000,0.00000,0.00000,0.00000,True,True
1,M2 log,0.00175,0.00169,0.00065,0.00057,True,True
2,M3 log+,0.00695,0.00661,0.00371,0.00257,True,True
3,M4 full,0.03051,0.02913,0.08531,0.05986,True,False
4,M5 amenities,0.00310,0.00285,0.00128,0.00060,True,True
5,M6 final,0.00110,0.00102,0.00046,0.00032,True,True


Видим, что у всех моделей есть признаки гетероскедатичности.

Поэтому обычные стандартные ошибки могут быть некорректными.

### 6. Проверка нормальности случайной ошибки

Для проверки нормальности остатков используем тест Жарка–Бера.

Гипотезы:

H_0: остатки распределены нормально.

H_1: остатки не распределены нормально.

Если p-value меньше 0.05, гипотеза о нормальности отвергается.

In [ ]:
from statsmodels.stats.stattools import jarque_bera

models = {
    'M1 price': model_1,
    'M2 log': model_2,
    'M3 log+': model_3,
    'M4 full': model_4,
    'M5 amenities': model_5,
    'M6 final': model_6
}

normality_results = []

for name, model in models.items():
    jb_stat, jb_pvalue, skew, kurtosis = jarque_bera(model.resid)

    normality_results.append({
        'model': name,
        'JB statistic': jb_stat,
        'p_value': jb_pvalue,
        'skewness': skew,
        'kurtosis': kurtosis,
        'reject_H0_5pct': jb_pvalue < 0.05
    })

normality_table = pd.DataFrame(normality_results)

normality_table.round(5)

,model,JB statistic,p_value,skewness,kurtosis,reject_H0_5pct
0,M1 price,4261.61667,0.0,2.43226,11.94217,True
1,M2 log,260.55782,0.0,0.98835,4.55833,True
2,M3 log+,306.62137,0.0,1.00544,4.84721,True
3,M4 full,330.08540,0.0,1.00815,4.99021,True
4,M5 amenities,291.06034,0.0,0.98351,4.79117,True
5,M6 final,276.27752,0.0,0.97640,4.70427,True


## Робастные стандартные ошибки для M6

Так как тесты Бреуша–Пагана и Уайта показали признаки гетероскедастичности, обычные стандартные ошибки могут быть некорректными. Поэтому для итоговой модели M6 дополнительно считаю робастные стандартные ошибки HC3.

In [ ]:
model_6_robust = model_6.get_robustcov_results(cov_type='HC3')

print(model_6_robust.summary())

                            OLS Regression Results                            
Dep. Variable:               ln_price   R-squared:                       0.523
Model:                            OLS   Adj. R-squared:                  0.519
Method:                 Least Squares   F-statistic:                     110.8
Date:                Wed, 06 May 2026   Prob (F-statistic):          1.88e-131
Time:                        15:18:24   Log-Likelihood:                -759.53
No. Observations:                 987   AIC:                             1537.
Df Residuals:                     978   BIC:                             1581.
Df Model:                           8                                         
Covariance Type:                  HC3                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             10.3224      0

## Финальная интерпретация модели M6

1. Модель в целом значима: p-value F-теста < 0.05.

2. R^2 = 0.523, adjusted R^2 = 0.519: модель объясняет около 52% различий в логарифме цены аренды.

3. Площадь значимо повышает цену: при росте площади на 1% цена растёт примерно на 0.30%.

4. Количество комнат также значимо повышает цену.

5. Расстояние до центра значимо снижает цену: +1 км от центра связан примерно с −1.2% к цене.

6. Апартаменты в среднем дороже обычных квартир примерно на 15.8%.

7. `repair_euro` и `amenities_count` значимы, но имеют отрицательные знаки, поэтому их нужно интерпретировать осторожно.

8. `relative_floor` и `utilities_included` после робастных ошибок незначимы на 5% уровне.


Уравнение модели M6:


$
\ln(price_i) =
10.3224
+ 0.2985 \ln(total_-meters_i)
+ 0.2293 rooms_-count_i
- 0.0119 center_-distance_i
- 0.1113 relative_-floor_i
+ 0.1468 is_-apartment_i
+ 0.0647 utilities_-included_i
- 0.0888 repair_-euro_i
- 0.0364 amenities_-count_i
+ u_i
$
